# Reproducing the source's original train/val/test split

Some of the datasets were published with an original split. Reproducing a paper's numbers, or comparing against them honestly, means using the split the paper used.

`AssignSourceSplit` is the transform that implements that. It does one thing: read the source's own label out of the `original_split` column and write a `train` / `val` / `test` value into the `split` column.

It needs configuring per dataset, since no two sources name their splits the same way. This notebook walks through every dataset in the **human communication** group that publishes a split, shows what each one actually says, and shows the mapping that turns it into something a collection can pool.

---
## Setup

In [1]:
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import neuralset as ns
from neuralshed import studies, transforms
from neuralshed.database import collections as dbc

pd.set_option("display.width", 170, "display.max_columns", 40)

MEMBERS = [
    "fan_2023",
    "kunz_2025",
    "wairagkar_2025",
    "card_2024",
    "willet_2023",
    "willet_2021",
]
PATHS = dbc.data_dirs(dataset_ids=MEMBERS)
COLUMNS = ["timeline", "type", "start", "duration", "text", "original_split", "split"]


def member_chain(study_class, mapping, query=None):
    """A member chain: the Study followed by its own AssignSourceSplit."""
    return ns.Chain(
        steps={
            study_class.__name__: study_class(
                path=str(PATHS[study_class.__name__]),
                query=query,
            ),
            "AssignSourceSplit": transforms.AssignSourceSplit(mapping=mapping),
        }
    )


def crosstab(events, target="Sentence"):
    """What the source published for each target row, against what the transform assigned."""
    targets = events[events["type"] == target]
    source = targets["original_split"].fillna("(absent)").replace("", "(no label)")
    assigned = targets["split"].replace("", "(unassigned)")
    return pd.crosstab(source, assigned)

## The `AssignSourceSplit` transform

Input arguments:

| field | default | what it is |
|---|---|---|
| `mapping` | **required** | how this source's labels translate into `train` / `val` / `test` |
| `source` | `original_split` | the column the study wrote its own label into |
| `dest` | `split` | the column the rest of the pipeline reads |

`mapping` deliberately has no default, since there is no vocabulary the six datasets below agree on.

Two rules cover everything the transform does:

- a label the mapping **does not name** comes out an empty string
- a row with **no label at all** comes out an empty string too

---
## 1. `fan_2023`

FALCON does not use the words train, val and test. It splits recording days into **held-in** and **held-out**, and the fan_2023 Study carries FALCON's label on every trial:

| FALCON name | what it is for |
|---|---|
| `held-in-calib` | the first 60% of each held-in day's trials, released for training a decoder |
| `held-in-eval` | the remaining 40%, withheld by FALCON to score submissions within the training days, public on Dryad |
| `held-out-calib` | the first three trials of each later day, for few-shot recalibration |
| `""` | the no-recalibration blocks, which FALCON did not use |

The mapping is a judgement call. `held-in-calib` is training. `held-in-eval` comes from the same days as training, so it serves as validation. `held-out-calib` is what a benchmark entry would *fine-tune* on before being scored on withheld trials we do not have. Calling it `test` keeps those later days out of training. The no-recalibration trials stay unassigned.

The Study also carries the authors' own partition as `corp_split` (the last 10 trials of each seed day are `test`), which a member could map instead.

The label sits on the trial's timeline, so both rows of a trial inherit it.

In [2]:
chain = member_chain(
    study_class=studies.Fan2023Handwriting,
    mapping={"held-in-calib": "train", "held-in-eval": "val", "held-out-calib": "test"},
)
events_fan = chain.run()
events_fan[COLUMNS]

,timeline,type,start,duration,text,original_split,split
0,"Fan2023Handwriting:block=14,block_kind=seed,co...",SpikeCounts,0.0,53.74,NaN,held-in-calib,train
1,"Fan2023Handwriting:block=14,block_kind=seed,co...",Sentence,0.0,53.74,there just might be too many people there.,held-in-calib,train
2,"Fan2023Handwriting:block=14,block_kind=seed,co...",SpikeCounts,0.0,50.72,NaN,held-in-calib,train
3,"Fan2023Handwriting:block=14,block_kind=seed,co...",Sentence,0.0,50.72,he was an important man in his village.,held-in-calib,train
4,"Fan2023Handwriting:block=14,block_kind=seed,co...",SpikeCounts,0.0,19.64,NaN,held-in-calib,train
...,...,...,...,...,...,...,...
2655,"Fan2023Handwriting:block=6,block_kind=recalibr...",Sentence,0.0,45.56,at least she never knew what the bottom was like.,held-out-calib,test
2656,"Fan2023Handwriting:block=6,block_kind=recalibr...",SpikeCounts,0.0,25.02,NaN,held-out-calib,test
2657,"Fan2023Handwriting:block=6,block_kind=recalibr...",Sentence,0.0,25.02,"strange, gabriel said.",held-out-calib,test
2658,"Fan2023Handwriting:block=6,block_kind=recalibr...",SpikeCounts,0.0,46.46,NaN,held-out-calib,test


In [3]:
crosstab(events_fan, target="Sentence")

split,(unassigned),test,train,val
original_split,,,,
(no label),199,0,0,0
held-in-calib,0,0,671,0
held-in-eval,0,0,0,445
held-out-calib,0,15,0,0


671 sentences for training, 445 for validation, 15 for testing, and 199 unassigned, the no-recalibration blocks FALCON left out.

---
## 2. `kunz_2025`

kunz marks a recording file `eval` when it was a closed-loop block, one where sentences were decoded live during the session. Everything else is `train`.

There is no validation set, so `train` and `eval` map straight onto `train` and `test`. Those eval trials differ from training data in more than having been held out: the participant was reacting to a decoder's output on screen.

The label belongs to a whole file, so it reaches every row.

In [4]:
chain = member_chain(
    study_class=studies.Kunz2025InnerSpeech,
    mapping={"train": "train", "eval": "test"},
)
events_kunz = chain.run()
events_kunz[COLUMNS]

,timeline,type,start,duration,text,original_split,split
0,"Kunz2025InnerSpeech:block=1,date=2024-12-06,ex...",SpikeCounts,0.00,474.00,NaN,train,train
1,"Kunz2025InnerSpeech:block=1,date=2024-12-06,ex...",BandPower,0.00,474.00,NaN,train,train
2,"Kunz2025InnerSpeech:block=1,date=2024-12-06,ex...",Stimulus,0.00,3.98,NaN,train,train
3,"Kunz2025InnerSpeech:block=1,date=2024-12-06,ex...",Sentence,3.98,9.14,chittychittybangbang You have glasses like that.,train,train
4,"Kunz2025InnerSpeech:block=1,date=2024-12-06,ex...",Stimulus,3.98,0.02,NaN,train,train
...,...,...,...,...,...,...,...
43184,"Kunz2025InnerSpeech:block=4,date=2024-07-17,ex...",Stimulus,268.08,0.04,NaN,,
43185,"Kunz2025InnerSpeech:block=4,date=2024-07-17,ex...",Intertrial,272.08,1.00,NaN,,
43186,"Kunz2025InnerSpeech:block=4,date=2024-07-17,ex...",Stimulus,273.08,2.00,NaN,,
43187,"Kunz2025InnerSpeech:block=4,date=2024-07-17,ex...",Action,275.08,4.00,NaN,,


In [5]:
crosstab(events_kunz, target="Sentence")

split,test,train
original_split,,
eval,370,0
train,0,1980


1,980 training sentences (34 files) 370 eval sentences (7 files).

---
## 3. `wairagkar_2025`

Same two labels as kunz, `train` and `eval`, and the same mapping, but a different granularity. Here a timeline is a single trial, so `original_split` identifies one trial rather than a whole session, and a session freely mixes both labels.

The label is a timeline key, so it can go straight into the study's query, which filters timelines before any data is read. The eval trials then never load at all:

```python
studies.Wairagkar2025Brain2Voice(path=..., query="original_split == 'train'")
```

This dataset was published for voice synthesis, but every trial also carries the sentence that was cued, so it can serve as a text target too.

In [6]:
chain = member_chain(
    study_class=studies.Wairagkar2025Brain2Voice,
    mapping={"train": "train", "eval": "test"},
)
events_wairagkar = chain.run()
events_wairagkar[COLUMNS]

,timeline,type,start,duration,text,original_split,split
0,"Wairagkar2025Brain2Voice:benchmark=,date=2023-...",BandPower,0.00,8.57,NaN,train,train
1,"Wairagkar2025Brain2Voice:benchmark=,date=2023-...",SpikeCounts,0.00,8.57,NaN,train,train
2,"Wairagkar2025Brain2Voice:benchmark=,date=2023-...",Sentence,0.00,8.57,How good are you?,train,train
3,"Wairagkar2025Brain2Voice:benchmark=,date=2023-...",AcousticFeatures,0.00,8.57,NaN,train,train
4,"Wairagkar2025Brain2Voice:benchmark=,date=2023-...",AcousticFeatures,0.00,8.57,NaN,train,train
...,...,...,...,...,...,...,...
95718,"Wairagkar2025Brain2Voice:benchmark=,date=2024-...",Word,15.04,0.78,in,eval,test
95719,"Wairagkar2025Brain2Voice:benchmark=,date=2024-...",Word,16.70,1.01,more,eval,test
95720,"Wairagkar2025Brain2Voice:benchmark=,date=2024-...",Word,18.33,0.94,than,eval,test
95721,"Wairagkar2025Brain2Voice:benchmark=,date=2024-...",Word,20.38,1.01,three,eval,test


In [7]:
crosstab(events_wairagkar, target="Sentence")

split,test,train
original_split,,
eval,1216,0
train,0,7273


7,273 training trials and 1,216 eval trials, each its own timeline with its own neural streams.

---
## 4. `card_2024`

card publishes `train`, `val` and `test`, but the target values of its `test` trials were held back for a public competition and their sentences were never released, so they have neural data and no target at all.

Mapping that label to `test` would create an evaluation split there is no way to score. So we leave it out of the mapping and those rows come through unassigned.

In [8]:
chain = member_chain(
    study_class=studies.Card2024Brain2Text,
    mapping={"train": "train", "val": "val"},
)
events_card = chain.run()
events_card[COLUMNS]

,timeline,type,start,duration,text,original_split,split
0,"Card2024Brain2Text:block=2,date=2023-08-11,ori...",SpikeCounts,0.0,6.42,NaN,train,train
1,"Card2024Brain2Text:block=2,date=2023-08-11,ori...",BandPower,0.0,6.42,NaN,train,train
2,"Card2024Brain2Text:block=2,date=2023-08-11,ori...",Sentence,0.0,6.42,Bring it closer.,train,train
3,"Card2024Brain2Text:block=2,date=2023-08-11,ori...",SpikeCounts,0.0,9.62,NaN,train,train
4,"Card2024Brain2Text:block=2,date=2023-08-11,ori...",BandPower,0.0,9.62,NaN,train,train
...,...,...,...,...,...,...,...
31389,"Card2024Brain2Text:block=8,date=2025-04-13,ori...",BandPower,0.0,22.84,NaN,test,
31390,"Card2024Brain2Text:block=8,date=2025-04-13,ori...",SpikeCounts,0.0,16.46,NaN,test,
31391,"Card2024Brain2Text:block=8,date=2025-04-13,ori...",BandPower,0.0,16.46,NaN,test,
31392,"Card2024Brain2Text:block=8,date=2025-04-13,ori...",SpikeCounts,0.0,20.60,NaN,test,


In [9]:
crosstab(events_card, target="Sentence")

split,train,val
original_split,,
train,8072,0
val,0,1426


In [10]:
pd.crosstab(events_card["original_split"], events_card["type"])

type,BandPower,Sentence,SpikeCounts
original_split,,,
test,1450,0,1450
train,8072,8072,8072
val,1426,1426,1426


`test` is missing from the crosstab entirely. Counting event types per label shows why: those 1,450 trials carry both neural streams and zero sentences, so there is no target row for the crosstab to count.

---
## 5. `willet_2023`

Willett 2023 stores one timeline per recording **block** with 20 to 50 sentences inside it, and the published split is per **block**: the last competition-eligible large-vocabulary block of each day is the test block and the earlier ones are training. So the split is a property of the timeline, and the study carries it on the timeline dict as `original_split`, broadcast to every row of the block, neural streams included.

Two consequences:

- the split could be pushed into the study's pre-loading query (`query="original_split == 'train'"`) the way wairagkar's can, although this notebook maps it with `AssignSourceSplit` like every other member
- the block's neural streams carry the same `split` value as its sentences, so a frame filtered on `split == "train"` keeps the brain data the sentences point at

There is also a third category. The authors left some blocks out of the partition entirely, mostly repeated material and the fifty-sentence set, and the study marks those blocks with an empty label.

In [11]:
chain = member_chain(
    study_class=studies.Willett2023Sentences,
    mapping={"train": "train", "test": "test"},
)
events_willet_2023 = chain.run()
events_willet_2023[COLUMNS]

,timeline,type,start,duration,text,original_split,split
0,"Willett2023Sentences:block=3,block_type=OL Ope...",SpikeCounts,0.00,382.24,NaN,train,train
1,"Willett2023Sentences:block=3,block_type=OL Ope...",BandPower,0.00,382.24,NaN,train,train
2,"Willett2023Sentences:block=3,block_type=OL Ope...",Stimulus,0.00,8.00,NaN,train,train
3,"Willett2023Sentences:block=3,block_type=OL Ope...",Sentence,8.00,9.58,Nuclear rockets can destroy airfields with ease.,train,train
4,"Willett2023Sentences:block=3,block_type=OL Ope...",Stimulus,8.00,0.02,NaN,train,train
...,...,...,...,...,...,...,...
33307,"Willett2023Sentences:block=18,block_type=CL Ch...",Sentence,446.68,2.68,It is okay,,
33308,"Willett2023Sentences:block=18,block_type=CL Ch...",Stimulus,446.68,0.02,NaN,,
33309,"Willett2023Sentences:block=18,block_type=CL Ch...",Stimulus,450.84,3.34,NaN,,
33310,"Willett2023Sentences:block=18,block_type=CL Ch...",Sentence,454.18,4.26,Bring my glasses here,,


Notice how, in the frame above, the neural streams carry the same split as their sentences, because the block is the unit of both. Where the label lives on the event row alone, as in `willet_2021` below, the streams sit unassigned, and filtering the whole frame by `split == "train"` keeps the sentences while dropping the brain data they point at. Selection then has to happen at the trigger, not across the frame.

In [12]:
crosstab(events_willet_2023, target="Sentence")

split,(unassigned),test,train
original_split,,,
(no label),1220,0,0
test,0,880,0
train,0,0,8800


8,800 train and 880 test, one held-out block per session, plus 1,220 sentences unassigned, about 11% of the total.

---
## 6. `willet_2021`

Willett 2021 has the same per-sentence shape as Willett 2023, but it ships two partition options:

| file | what it holds out |
|---|---|
| `trainTestPartitions_HeldOutBlocks.mat` | whole blocks |
| `trainTestPartitions_HeldOutTrials.mat` | individual trials scattered through blocks that stay in training |

We default to whole blocks, set by `_PARTITION` in the study module, because a test sentence sitting inside a training block shares that block's drift and normalization statistics with its neighbours. Whole blocks is also what Willett 2023 does, so the two behave the same way in one collection.

Some trials are kept unassigned: 33 sentences the authors flagged as bad, plus 237 trials where only one of the three task conditions was partitioned at all.

In [13]:
chain = member_chain(
    study_class=studies.Willett2021Handwriting,
    query="task == 'sentences'",
    mapping={"train": "train", "test": "test"},
)
events_willet_2021 = chain.run()
events_willet_2021[COLUMNS]

,timeline,type,start,duration,text,original_split,split
0,"Willett2021Handwriting:block=6,date=2019-05-08...",SpikeCounts,0.00,759.09,NaN,NaN,
1,"Willett2021Handwriting:block=6,date=2019-05-08...",Stimulus,18.67,8.66,NaN,NaN,
2,"Willett2021Handwriting:block=6,date=2019-05-08...",Sentence,27.33,19.61,"thanks so much, honey.",train,train
3,"Willett2021Handwriting:block=6,date=2019-05-08...",Stimulus,27.33,0.01,NaN,NaN,
4,"Willett2021Handwriting:block=6,date=2019-05-08...",Stimulus,46.95,8.94,NaN,NaN,
...,...,...,...,...,...,...,...
3089,"Willett2021Handwriting:block=18,date=2020-01-1...",Sentence,363.93,27.45,bigger,,
3090,"Willett2021Handwriting:block=18,date=2020-01-1...",Stimulus,363.93,0.01,NaN,NaN,
3091,"Willett2021Handwriting:block=18,date=2020-01-1...",Stimulus,391.39,1.00,NaN,NaN,
3092,"Willett2021Handwriting:block=18,date=2020-01-1...",Sentence,392.39,60.72,i like to drive late at night so there are few...,,


In [14]:
crosstab(events_willet_2021, target="Sentence")

split,(unassigned),test,train
original_split,,,
(no label),269,0,0
test,0,228,0
train,0,0,502


Same shape as Willett 2023, with a much larger unassigned group: 27% of sentences against Willett 2023's 11%.

---
## Summary

| member | publishes | mapping | label on | train / val / test | unassigned |
|---|---|---|---|---|---|
| `fan_2023` | `held-in-calib` / `held-in-eval` / `held-out-calib` | → train / val / test | trial | 671 / 445 / 15 | 199 |
| `kunz_2025` | `train` / `eval` | → train / test | file | 1,980 / - / 370 | 0 |
| `wairagkar_2025` | `train` / `eval` | → train / test | trial | 7,273 / - / 1,216 | 0 |
| `card_2024` | `train` / `val` / `test` | → train / val, test omitted | trial | 8,072 / 1,426 / - | 0 |
| `willet_2023` | `train` / `test` | → train / test | block | 8,800 / - / 880 | 1,220 |
| `willet_2021` | `train` / `test` | → train / test | sentence | 502 / - / 228 | 269 |

Four things this leaves open:

**Unassigned rows:** the 199, 1,220 and 269 are not one thing. fan_2023's are the no-recalibration blocks, which FALCON left out of its benchmark. Willett 2021's are 32 sentences the authors flagged as bad, which should stay out, plus 237 good trials from a task condition they never partitioned. Willett 2023's are all usable sentences the competition didn't use. Recovering them needs a fallback that does not exist yet, e.g.: running `AssignHashSplit` afterwards replaces the published split rather than filling its gaps.

**Validation:** only card_2024 publishes a validation split. fan_2023's comes from mapping FALCON's within-day evaluation trials to it.

**wairagkar's** 1,216 eval trials include the 90 `random` and 37 `video` trials the paper actually scored for intelligibility, which the study already carries in a `benchmark` column. Mapping everything to `test` loses that distinction.

**Split granularity:** session, file, trial and block are timeline keys that reach every row, sentence (willet_2021) is an event row that does not. Downstream already covers for that, since `valid_targets()` filters on `target_is_valid` before it reads `split` and the bundle, preprocessing and validator all start from there.